In [ ]:
%run ./1_config


In [ ]:

import json, os, time, urllib.request
from pathlib import Path
from pyspark.sql import functions as F

class SilverGeoLoader:
    def __init__(self):
        self.c = conf

    def distinct_ips(self):
        return [r.ip for r in spark.table(self.c.table_fqn(self.c.bronze_events)).select("ip").distinct().collect() if r.ip]

    def load_mock(self, ip):
        for p in [Path("data/fixtures/ipstack") / f"{ip}.json", Path(f"/lakehouse/{self.c.lakehouse}/Files/fixtures/ipstack/{ip}.json")]:
            if p.exists():
                return p.read_text(), 200
        return json.dumps({"ip": ip, "country_code": "XX", "country_name": "Unknown", "connection": {"isp": "Unknown"}, "security": {"is_proxy": false, "threat_level": "low"}}), 200

    def load_live(self, ip):
        key = os.getenv("IPSTACK_ACCESS_KEY", "")
        if not key:
            raise ValueError("IPSTACK_ACCESS_KEY required when MOCK_IPSTACK=false")
        url = f"http://api.ipstack.com/{ip}?access_key={key}&security=1"
        with urllib.request.urlopen(url, timeout=15) as resp:
            return resp.read().decode(), resp.getcode()

    def write_bronze_api(self, records):
        df = spark.createDataFrame(records, "ip STRING, response_json STRING, http_status INT, api_called_at TIMESTAMP")
        df.write.format("delta").mode("append").saveAsTable(self.c.table_fqn(self.c.bronze_ipstack_raw))

    def enrich(self, ips):
        rows = []
        for ip in ips:
            try:
                body, status = self.load_mock(ip) if self.c.mock_ipstack else self.load_live(ip)
                if not self.c.mock_ipstack:
                    time.sleep(0.25)
                rows.append((ip, body, int(status)))
            except Exception as e:
                err = spark.createDataFrame([(ip, str(e), 0)], "ip STRING, error_message STRING, http_status INT")
                err = err.withColumn("api_called_at", F.current_timestamp())
                err.write.format("delta").mode("append").saveAsTable(self.c.table_fqn(self.c.bronze_ipstack_errors))
        if rows:
            df = spark.createDataFrame(rows, "ip STRING, response_json STRING, http_status INT")
            df = df.withColumn("api_called_at", F.current_timestamp())
            df.write.format("delta").mode("append").saveAsTable(self.c.table_fqn(self.c.bronze_ipstack_raw))

    def ip_dim(self):
        raw = spark.table(self.c.table_fqn(self.c.bronze_ipstack_raw))
        j = "ip STRING, country_code STRING, country_name STRING, region_name STRING, city STRING, latitude DOUBLE, longitude DOUBLE, time_zone STRUCT<id:STRING>, currency STRUCT<code:STRING>, connection STRUCT<isp:STRING,org:STRING>, security STRUCT<is_proxy:BOOLEAN,proxy_type:STRING,threat_level:STRING>"
        p = raw.withColumn("j", F.from_json("response_json", j))
        dim = p.select(
            "ip",
            F.col("j.country_code").alias("country_code"),
            F.col("j.country_name").alias("country_name"),
            F.col("j.region_name").alias("region"),
            F.col("j.city").alias("city"),
            F.col("j.latitude").alias("latitude"),
            F.col("j.longitude").alias("longitude"),
            F.col("j.time_zone.id").alias("timezone_id"),
            F.col("j.currency.code").alias("currency_code"),
            F.col("j.connection.isp").alias("isp"),
            F.col("j.connection.org").alias("org"),
            F.coalesce(F.col("j.security.is_proxy"), F.lit(False)).alias("is_proxy"),
            F.when(F.lower(F.coalesce(F.col("j.security.proxy_type"), F.lit(""))).contains("vpn"), True).otherwise(False).alias("is_vpn"),
            F.col("j.security.threat_level").alias("threat_level"),
            F.current_timestamp().alias("enriched_at"),
        ).dropDuplicates(["ip"])
        dim.write.format("delta").mode("overwrite").saveAsTable(self.c.table_fqn(self.c.silver_ip_dim))

    def enriched_events(self):
        ev = spark.table(self.c.table_fqn(self.c.bronze_events))
        dim = spark.table(self.c.table_fqn(self.c.silver_ip_dim))
        device = F.when(F.col("user_agent").contains("iPhone"), F.lit("mobile")).otherwise(F.lit("desktop"))
        out = (ev.join(dim, "ip", "left")
               .withColumn("device", device)
               .withColumn("spend", F.col("amount"))
               .select("event_id","session_id","event_ts","event_type","user_id","ip","user_agent","amount","event_date",
                       "country_code","country_name","timezone_id","isp","is_vpn","device","spend"))
        out.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(self.c.table_fqn(self.c.silver_events_enriched))

    def run(self):
        ips = self.distinct_ips()
        print(f"IPs to enrich: {len(ips)} mock={self.c.mock_ipstack}")
        self.enrich(ips)
        self.ip_dim()
        self.enriched_events()
        print("✅ Silver complete")

SilverGeoLoader().run()

